[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/13_code_search.ipynb)

# 13. Code Search

**Part 3: Context** · about 25 minutes

The cheapest token is the one you never read. Notebook 11 showed how one big file can take over the context. Code search is the positive version of that lesson: give the agent ways to **find** the right lines, so it doesn't have to read everything to locate them.

When you join a new codebase you don't read every file top to bottom. You look at the folder structure, search for a word, jump to a definition, and check where it's used. This notebook gives the agent those same moves:

| Tool | Answers | Built with |
|---|---|---|
| `tree` | What's in this project? | the filesystem |
| `grep` | Where does this text appear? | regular expressions |
| `outline` | What's in this file, without reading it all? | Python's `ast` module |
| `find_definition` | Where is this function or class defined? | `ast` |
| `find_references` | Who uses it? | `ast` |

The last three *understand* Python rather than matching text, so they aren't fooled by comments, strings or names that merely contain the word.

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import ast
import json
import re
import shutil
import tempfile
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. A real-looking project

A small online shop: models, pricing with GST and discounts, inventory, orders, receipts and reports, plus tests. It's 10 files, small enough to read here, but already enough that reading everything to answer one question is wasteful.

In [3]:
SHOP = {
    "shop/__init__.py": '"""A small online shop: products, orders, pricing and receipts."""\n',
    "shop/models.py": '''\
from dataclasses import dataclass, field


@dataclass
class Product:
    sku: str
    name: str
    price: float  # in rupees


@dataclass
class Customer:
    name: str
    email: str
    years: int = 0  # how long they've been a customer


@dataclass
class Order:
    customer: Customer
    lines: list = field(default_factory=list)  # (Product, quantity) pairs

    def add(self, product, quantity=1):
        if quantity < 1:
            raise ValueError("quantity must be at least 1")
        self.lines.append((product, quantity))
''',
    "shop/pricing.py": '''\
"""Prices, discounts and tax."""

GST_RATE = 0.18


def subtotal(order):
    """Sum of price times quantity for every line."""
    return sum(product.price * quantity for product, quantity in order.lines)


def loyalty_discount(amount, years):
    """2% off for every year as a customer, capped at 10%."""
    rate = min(years * 2, 10)
    return round(amount * rate / 100, 2)


def coupon_discount(amount, code):
    """Flat discounts for known coupon codes, never more than the amount."""
    coupons = {"WELCOME50": 50, "DIWALI200": 200}
    return min(coupons.get(code, 0), amount)


def final_price(order, coupon=None):
    """What the customer pays, including GST."""
    amount = subtotal(order)
    amount = amount - loyalty_discount(amount, order.customer.years)
    if coupon:
        amount -= coupon_discount(amount, coupon)
    return round(amount * (1 + GST_RATE), 2)
''',
    "shop/inventory.py": '''\
"""Stock levels and reservations."""

STOCK = {}


def set_stock(sku, quantity):
    STOCK[sku] = quantity


def available(sku):
    return STOCK.get(sku, 0)


def reserve(order):
    """Take stock for every line in the order, or nothing if any line is short."""
    for product, quantity in order.lines:
        if available(product.sku) < quantity:
            raise ValueError(f"not enough stock for {product.name}")
    for product, quantity in order.lines:
        STOCK[product.sku] -= quantity


def release(order):
    """Put an order's stock back, for example after a cancellation."""
    for product, quantity in order.lines:
        STOCK[product.sku] = available(product.sku) + quantity
''',
    "shop/orders.py": '''\
"""Placing and cancelling orders."""

from shop import inventory, notifications
from shop.pricing import final_price

PLACED = []


def place_order(order, coupon=None):
    inventory.reserve(order)
    total = final_price(order, coupon)
    PLACED.append((order, total))
    notifications.send_receipt(order, total)
    return total


def cancel_order(order):
    for index, (placed, _) in enumerate(PLACED):
        if placed is order:
            PLACED.pop(index)
            inventory.release(order)
            return True
    return False
''',
    "shop/notifications.py": '''\
"""Customer messages. Returns text instead of sending email, to keep things testable."""

from shop.utils import format_rupees

OUTBOX = []


def send_receipt(order, total):
    lines = [f"Hi {order.customer.name}, thanks for your order!"]
    for product, quantity in order.lines:
        lines.append(f"  {quantity} x {product.name}")
    lines.append(f"Total paid: {format_rupees(total)}")
    message = "\\n".join(lines)
    OUTBOX.append((order.customer.email, message))
    return message
''',
    "shop/reports.py": '''\
"""Numbers for the shop owner."""

from shop.pricing import final_price, loyalty_discount, subtotal


def daily_revenue(placed):
    return round(sum(total for _, total in placed), 2)


def loyalty_savings(orders):
    """How much loyal customers saved before tax, in total."""
    return round(sum(loyalty_discount(subtotal(o), o.customer.years) for o in orders), 2)


def biggest_order(orders):
    return max(orders, key=final_price, default=None)
''',
    "shop/utils.py": '''\
"""Small helpers."""


def format_rupees(amount):
    """Format with Indian digit grouping, like Rs 1,23,456.50."""
    rupees, paise = f"{amount:.2f}".split(".")
    if len(rupees) > 3:
        head, tail = rupees[:-3], rupees[-3:]
        groups = []
        while len(head) > 2:
            groups.insert(0, head[-2:])
            head = head[:-2]
        rupees = ",".join([head] + groups + [tail]) if head else ",".join(groups + [tail])
    return f"Rs {rupees}.{paise}"
''',
    "tests/test_pricing.py": '''\
import unittest

from shop.models import Customer, Order, Product
from shop.pricing import coupon_discount, final_price, loyalty_discount, subtotal

TEA = Product("T1", "Masala tea", 100.0)


class PricingTests(unittest.TestCase):
    def test_subtotal(self):
        order = Order(Customer("Asha", "asha@example.com"))
        order.add(TEA, 3)
        self.assertEqual(subtotal(order), 300.0)

    def test_loyalty_is_capped(self):
        self.assertEqual(loyalty_discount(1000, 2), 40.0)
        self.assertEqual(loyalty_discount(1000, 20), 100.0)

    def test_loyal_customers_pay_less(self):
        new, loyal = Order(Customer("A", "a@example.com", 0)), Order(Customer("B", "b@example.com", 5))
        new.add(TEA, 10)
        loyal.add(TEA, 10)
        self.assertLess(final_price(loyal), final_price(new))
        self.assertEqual(final_price(loyal), 1062.0)

    def test_coupon_never_exceeds_amount(self):
        self.assertEqual(coupon_discount(30, "WELCOME50"), 30)
''',
    "tests/test_utils.py": '''\
import unittest

from shop.utils import format_rupees


class FormatTests(unittest.TestCase):
    def test_small(self):
        self.assertEqual(format_rupees(950), "Rs 950.00")

    def test_indian_grouping(self):
        self.assertEqual(format_rupees(123456.5), "Rs 1,23,456.50")
''',
}

workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()
for path, text in SHOP.items():
    (workspace / path).parent.mkdir(parents=True, exist_ok=True)
    (workspace / path).write_text(text)

everything = sum(len(text) for text in SHOP.values())
print(f"{len(SHOP)} files, {sum(len(t.splitlines()) for t in SHOP.values())} lines, {everything:,} characters")

10 files, 188 lines, 5,414 characters


## 2. `tree` and `grep`

`tree` is the agent's first look around. Line counts help it guess which files matter and which are big. `grep` uses regular expressions and is case-insensitive, which suits how models search ("loyal" should find `Loyalty` and `loyal_customer`).

In [4]:
def python_files():
    for file in sorted(workspace.rglob("*.py")):
        yield file.relative_to(workspace).as_posix(), file.read_text()


def tree():
    """Every Python file with its line count."""
    return "\n".join(f"{path}  ({len(text.splitlines())} lines)" for path, text in python_files())


def grep(pattern):
    """Lines matching a regular expression (case-insensitive), as file:line: text."""
    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error as error:
        return f"Error: invalid regular expression ({error}). Escape special characters like ( or ."
    hits = [f"{path}:{number}: {line.strip()}"
            for path, text in python_files()
            for number, line in enumerate(text.splitlines(), 1) if regex.search(line)]
    if not hits:
        return f"No matches for {pattern!r}"
    return "\n".join(hits[:40]) + (f"\n... and {len(hits) - 40} more" if len(hits) > 40 else "")


print(tree())
print()
print(grep("loyal"))

shop/__init__.py  (1 lines)
shop/inventory.py  (26 lines)
shop/models.py  (26 lines)
shop/notifications.py  (15 lines)
shop/orders.py  (23 lines)
shop/pricing.py  (29 lines)
shop/reports.py  (16 lines)
shop/utils.py  (14 lines)
tests/test_pricing.py  (27 lines)
tests/test_utils.py  (11 lines)

shop/pricing.py:11: def loyalty_discount(amount, years):
shop/pricing.py:26: amount = amount - loyalty_discount(amount, order.customer.years)
shop/reports.py:3: from shop.pricing import final_price, loyalty_discount, subtotal
shop/reports.py:10: def loyalty_savings(orders):
shop/reports.py:11: """How much loyal customers saved before tax, in total."""
shop/reports.py:12: return round(sum(loyalty_discount(subtotal(o), o.customer.years) for o in orders), 2)
tests/test_pricing.py:4: from shop.pricing import coupon_discount, final_price, loyalty_discount, subtotal
tests/test_pricing.py:15: def test_loyalty_is_capped(self):
tests/test_pricing.py:16: self.assertEqual(loyalty_discount(1000, 2), 40.0)
te

`grep` found every line that mentions the word, in the code and the tests: the definition, a docstring, imports, test names and variables called `loyal`. A text search can't tell a definition from a mention. For that we need to read the code the way Python does.

## 3. Reading code like Python does

Python's `ast` module turns source code into a tree of objects: function definitions, calls, names and so on, each with its line numbers. With it we can answer structural questions precisely.

First, `outline`: a file's table of contents. It's often all an agent needs to decide whether to read a file at all.

In [5]:
def first_doc_line(node):
    doc = ast.get_docstring(node)
    return f"  # {doc.splitlines()[0]}" if doc else ""


def outline(path):
    """Classes, functions and constants in a file, with line ranges and the first docstring line."""
    try:
        source = (workspace / path).read_text()
        module = ast.parse(source)
    except (OSError, SyntaxError) as error:
        return f"Error: {error}"
    lines = []
    for node in module.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            arguments = ", ".join(a.arg for a in node.args.args)
            lines.append(f"{node.lineno}-{node.end_lineno}  def {node.name}({arguments}){first_doc_line(node)}")
        elif isinstance(node, ast.ClassDef):
            lines.append(f"{node.lineno}-{node.end_lineno}  class {node.name}{first_doc_line(node)}")
            for item in node.body:
                if isinstance(item, ast.FunctionDef):
                    lines.append(f"    {item.lineno}-{item.end_lineno}  def {item.name}(...)")
        elif isinstance(node, ast.Assign) and all(isinstance(t, ast.Name) for t in node.targets):
            names = ", ".join(t.id for t in node.targets)
            lines.append(f"{node.lineno}  {names} = ...")
    return f"{path}\n" + "\n".join(lines)


print(outline("shop/pricing.py"))

shop/pricing.py
3  GST_RATE = ...
6-8  def subtotal(order)  # Sum of price times quantity for every line.
11-14  def loyalty_discount(amount, years)  # 2% off for every year as a customer, capped at 10%.
17-20  def coupon_discount(amount, code)  # Flat discounts for known coupon codes, never more than the amount.
23-29  def final_price(order, coupon)  # What the customer pays, including GST.


Now the two moves you use most in an editor: **go to definition** and **find all references**.

`find_references` reports the function each use sits in. That's the most useful part. "Called on line 30 of `pricing.py`" means little until you know that line 30 is inside `final_price`.

In [6]:
def definitions():
    for path, text in python_files():
        try:
            module = ast.parse(text)
        except SyntaxError:
            continue
        for node in ast.walk(module):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                yield path, text, node


def find_definition(name):
    """Where a function or class is defined, with its line range and first line."""
    found = [f"{path}:{node.lineno}-{node.end_lineno}: {text.splitlines()[node.lineno - 1].strip()}"
             for path, text, node in definitions() if node.name == name]
    return "\n".join(found) or f"No definition of {name!r} found."


def enclosing_function(module, line):
    best = None
    for node in ast.walk(module):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.lineno <= line <= node.end_lineno:
            if best is None or node.lineno > best.lineno:  # the innermost one wins
                best = node
    return best.name if best else "module level"


def find_references(name):
    """Every use of a name, with the function it appears in. Definitions themselves are left out."""
    found = []
    for path, text in python_files():
        try:
            module = ast.parse(text)
        except SyntaxError:
            continue
        lines = text.splitlines()
        for node in ast.walk(module):
            if isinstance(node, ast.ImportFrom) and any(alias.name == name for alias in node.names):
                found.append(f"{path}:{node.lineno} (import): {lines[node.lineno - 1].strip()}")
            elif (isinstance(node, ast.Name) and node.id == name) or \
                 (isinstance(node, ast.Attribute) and node.attr == name):
                where = enclosing_function(module, node.lineno)
                found.append(f"{path}:{node.lineno} (in {where}): {lines[node.lineno - 1].strip()}")
    return "\n".join(sorted(set(found))) or f"No uses of {name!r} found."


print(find_definition("loyalty_discount"))
print()
print(find_references("loyalty_discount"))

shop/pricing.py:11-14: def loyalty_discount(amount, years):

shop/pricing.py:26 (in final_price): amount = amount - loyalty_discount(amount, order.customer.years)
shop/reports.py:12 (in loyalty_savings): return round(sum(loyalty_discount(subtotal(o), o.customer.years) for o in orders), 2)
shop/reports.py:3 (import): from shop.pricing import final_price, loyalty_discount, subtotal
tests/test_pricing.py:16 (in test_loyalty_is_capped): self.assertEqual(loyalty_discount(1000, 2), 40.0)
tests/test_pricing.py:17 (in test_loyalty_is_capped): self.assertEqual(loyalty_discount(1000, 20), 100.0)
tests/test_pricing.py:4 (import): from shop.pricing import coupon_discount, final_price, loyalty_discount, subtotal


In [7]:
# Checkpoint: the structural tools must agree with what we know about the project.
assert find_definition("loyalty_discount").startswith("shop/pricing.py:")
refs = find_references("loyalty_discount")
assert "(in final_price)" in refs and "(in loyalty_savings)" in refs
assert "def loyalty_discount" not in refs  # the definition isn't a use
assert "No definition" in find_definition("does_not_exist")
print("Checkpoint passed.")

Checkpoint passed.


Compare this with `grep`. The references list skips the docstring and the definition line, and it tells you that in the shop code `loyalty_discount` is used by exactly two functions, `final_price` and `loyalty_savings`, plus the tests. A text search can't give you that.

These tools are deliberately simple. They work on names alone, so two different functions called `save` in different classes would look the same. Real code-intelligence tools, like the language servers inside editors, resolve exactly which object a name refers to. For a lot of agent work, simple and fast beats perfect, and you can always upgrade later.

## 4. The agent explores

We give the agent all five tools and ask a question that needs several of them. To keep this notebook about *finding* things, there's no shell and no editing.

In [8]:
def tool(tool_name, description, /, **properties):
    return {"type": "function", "function": {"name": tool_name, "description": description, "parameters": {
        "type": "object", "properties": {k: {"type": "string", "description": v} for k, v in properties.items()},
        "required": list(properties)}}}


TOOLS = [
    tool("tree", "List every Python file in the project with its line count."),
    tool("grep", "Search all Python files with a case-insensitive regular expression.", pattern="The regex"),
    tool("outline", "Show a file's classes, functions and constants with line numbers, without the full code.",
         path="File path"),
    tool("find_definition", "Find where a function or class is defined.", name="Exact function or class name"),
    tool("find_references", "Find every use of a function or class, and which function each use is in.",
         name="Exact function or class name"),
]
REGISTRY = {"tree": tree, "grep": grep, "outline": outline,
            "find_definition": find_definition, "find_references": find_references}


def run_agent(question, max_steps=10):
    messages = [
        {"role": "system", "content": "You answer questions about a codebase. Use the search tools to find exact "
                                      "locations instead of guessing. Give file names and line numbers."},
        {"role": "user", "content": question},
    ]
    read = 0
    for step in range(1, max_steps + 1):
        payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
        response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
        response.raise_for_status()
        message = response.json()["choices"][0]["message"]
        messages.append(message)
        if not message.get("tool_calls"):
            return {"answer": message.get("content"), "chars_read": read, "steps": step}
        for call in message["tool_calls"]:
            name = call["function"]["name"]
            try:
                arguments = json.loads(call["function"]["arguments"] or "{}")
                result = REGISTRY[name](**arguments) if name in REGISTRY else f"Error: unknown tool {name!r}"
            except Exception as error:
                result = f"Error: {type(error).__name__}: {error}"
            read += len(result)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            print(f"[{step}] {name}({call['function']['arguments']})")
            for line in result.splitlines()[:4]:
                print(f"     {line}")
    return {"answer": None, "chars_read": read, "steps": max_steps}


run = run_agent("Where is the discount for loyal customers calculated, and which functions use it?")
print("\nAnswer:", run["answer"])

[1] grep({"pattern": "loyal"})
     shop/pricing.py:11: def loyalty_discount(amount, years):
     shop/pricing.py:26: amount = amount - loyalty_discount(amount, order.customer.years)
     shop/reports.py:3: from shop.pricing import final_price, loyalty_discount, subtotal
     shop/reports.py:10: def loyalty_savings(orders):


[2] find_definition({"name": "loyalty_discount"})
     shop/pricing.py:11-14: def loyalty_discount(amount, years):


[3] find_references({"name": "loyalty_discount"})
     shop/pricing.py:26 (in final_price): amount = amount - loyalty_discount(amount, order.customer.years)
     shop/reports.py:12 (in loyalty_savings): return round(sum(loyalty_discount(subtotal(o), o.customer.years) for o in orders), 2)
     shop/reports.py:3 (import): from shop.pricing import final_price, loyalty_discount, subtotal
     tests/test_pricing.py:16 (in test_loyalty_is_capped): self.assertEqual(loyalty_discount(1000, 2), 40.0)



Answer: The loyalty discount is calculated by loyalty_discount in shop/pricing.py, lines 11-14. It's used by: final_price, loyalty_savings, test_loyalty_is_capped.


In [9]:
print(f"Characters the agent read through search tools: {run['chars_read']:,}")
print(f"Characters in the whole project:                {everything:,}")
print(f"That's {100 * run['chars_read'] / everything:.0f}% of the project.")

if llm.offline:
    assert "final_price" in run["answer"] and "loyalty_savings" in run["answer"]
    assert run["chars_read"] < everything / 2
print("Checkpoint passed.")

Characters the agent read through search tools: 1,871
Characters in the whole project:                5,414
That's 35% of the project.
Checkpoint passed.


The agent answered with exact locations after reading a small fraction of the code. On a 10-file project that saves a few thousand tokens. On a real codebase with thousands of files, reading everything isn't even possible, so search tools aren't an optimisation there. They're the only way the agent can work at all.

## Recap

- Search before you read. The cheapest token is the one you never load.
- `grep` finds text. `ast`-based tools find *code*: definitions, uses, and the function each use is in.
- An `outline` lets the agent decide whether a file is worth reading.
- Simple name-based tools are good enough for a lot of agent work, as long as you know their limits.

## Exercises

1. Add a `context` option to `grep` that shows N lines around each match. When is that better than a separate `read_file`?
2. `find_references` can't tell `Order.add` from any other method called `add`. Find a case in this project where that gives a misleading answer.
3. Write `callers_tree(name, depth=2)`: who calls this, and who calls *them*? Use it to answer "what could break if I change `subtotal`?"

## Clean up

In [10]:
shutil.rmtree(workspace)
print("Removed the temporary workspace.")

Removed the temporary workspace.


**Next:** [14. Memory](14_memory.ipynb). Making the agent remember what it learned, between runs.